In [ ]:
!pip install lightgbm scipy xgboost torch -q

In [ ]:
from google.colab import files

# This will open a file picker dialog
uploaded = files.upload()

# The uploaded file will be saved in the current directory
# Make sure you upload your CSV as 'stock_data.csv'
# Then run the pipeline:
# results = run_pipeline('stock_data.csv', forecast_horizon=63)


In [ ]:
"""
Stock Price Prediction (3-6 Months Ahead) — V2 (Improved)
==========================================================
Ensemble of Bi-LSTM with Attention + XGBoost + LightGBM.

Key improvements over V1:
  - All features are stationary (ratios/normalized, no raw prices)
  - Bi-directional LSTM with self-attention mechanism
  - Target scaling for LSTM
  - LightGBM added as third model
  - Purged walk-forward cross-validation
  - Stacking meta-learner ensemble
  - Huber loss for robustness to outliers
  - Multiple forecast horizons averaged for stability

Usage:
    1. Place CSV with columns [Date, Open, High, Low, Close, Volume] as 'stock_data.csv'
    2. Run: python stock_prediction.py
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM not installed. Install with: pip install lightgbm")


# ─────────────────────────────────────────────
# 0. CSV LOADING & COLUMN NORMALIZATION
# ─────────────────────────────────────────────

# Mapping of common alternative column names to standard names
COLUMN_ALIASES = {
    # Date
    'date': 'Date', 'datetime': 'Date', 'timestamp': 'Date', 'time': 'Date',
    'dlycaldt': 'Date',
    # Open
    'open': 'Open', 'dlyopen': 'Open', 'open_price': 'Open',
    # High
    'high': 'High', 'dlyhigh': 'High', 'high_price': 'High',
    # Low
    'low': 'Low', 'dlylow': 'Low', 'low_price': 'Low',
    # Close
    'close': 'Close', 'dlyclose': 'Close', 'close_price': 'Close',
    'adj close': 'Close', 'adj_close': 'Close', 'adjclose': 'Close',
    # Volume
    'volume': 'Volume', 'dlyvol': 'Volume', 'vol': 'Volume',
}


def normalize_columns(df):
    """Rename columns to standard names (Date, Open, High, Low, Close, Volume)."""
    rename_map = {}
    for col in df.columns:
        standard = COLUMN_ALIASES.get(col.lower().strip())
        if standard and standard not in rename_map.values():
            rename_map[col] = standard
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def load_csv_with_date(csv_path):
    """Load CSV, auto-detect columns, and parse dates correctly."""
    df = pd.read_csv(csv_path)
    df = normalize_columns(df)

    if 'Date' not in df.columns:
        # Try first column as date
        first_col = df.columns[0]
        try:
            pd.to_datetime(df[first_col].iloc[:5], dayfirst=True)
            df = df.rename(columns={first_col: 'Date'})
        except (ValueError, TypeError):
            # Fall back to index
            df = pd.read_csv(csv_path, index_col=0)
            df = normalize_columns(df)
            df.index = pd.to_datetime(df.index, dayfirst=True)
            df = df.reset_index()
            df = df.rename(columns={df.columns[0]: 'Date'})
            return df

    # Parse date with dayfirst=True to handle DD/MM/YYYY format correctly
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

    # Drop extra columns not needed (e.g. permno, dlycap, dlyretx, etc.)
    required = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    extra_cols = [c for c in df.columns if c not in required]
    if extra_cols:
        df = df.drop(columns=extra_cols)

    return df


# ─────────────────────────────────────────────
# 1. FEATURE ENGINEERING (ALL STATIONARY)
# ─────────────────────────────────────────────

def add_technical_indicators(df):
    """Engineer ONLY stationary/normalized features — no raw price levels."""
    df = df.copy()

    # --- Returns ---
    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['HL_Range'] = (df['High'] - df['Low']) / df['Close']
    df['OC_Range'] = (df['Close'] - df['Open']) / df['Close']

    # --- Price relative to Moving Averages (ratios, not raw prices) ---
    for window in [5, 10, 20, 50, 100, 200]:
        sma = df['Close'].rolling(window).mean()
        df[f'Close_to_SMA_{window}'] = df['Close'] / sma - 1
        # SMA slope (normalized)
        df[f'SMA_{window}_Slope'] = sma.pct_change(5)

    for span in [12, 26, 50]:
        ema = df['Close'].ewm(span=span).mean()
        df[f'Close_to_EMA_{span}'] = df['Close'] / ema - 1

    # --- MACD (already stationary as difference of EMAs relative to price) ---
    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['MACD_Norm'] = (ema12 - ema26) / df['Close']
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    df['MACD_Hist_Norm'] = (macd - signal) / df['Close']

    # --- RSI (already bounded 0-100) ---
    for period in [7, 14, 21]:
        delta = df['Close'].diff()
        gain = delta.clip(lower=0).rolling(period).mean()
        loss = (-delta.clip(upper=0)).rolling(period).mean()
        rs = gain / (loss + 1e-10)
        df[f'RSI_{period}'] = 100 - (100 / (1 + rs))

    # --- Bollinger Band Position (0-1 bounded) ---
    for window in [20, 50]:
        sma = df['Close'].rolling(window).mean()
        std = df['Close'].rolling(window).std()
        upper = sma + 2 * std
        lower = sma - 2 * std
        df[f'BB_Width_{window}'] = (upper - lower) / sma
        df[f'BB_Position_{window}'] = (df['Close'] - lower) / (upper - lower + 1e-10)

    # --- ATR (normalized) ---
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift(1)).abs()
    low_close = (df['Low'] - df['Close'].shift(1)).abs()
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    for period in [7, 14, 21]:
        df[f'ATR_Norm_{period}'] = true_range.rolling(period).mean() / df['Close']

    # --- Stochastic Oscillator (already bounded) ---
    for period in [14, 21]:
        low_n = df['Low'].rolling(period).min()
        high_n = df['High'].rolling(period).max()
        df[f'Stoch_K_{period}'] = 100 * (df['Close'] - low_n) / (high_n - low_n + 1e-10)
        df[f'Stoch_D_{period}'] = df[f'Stoch_K_{period}'].rolling(3).mean()

    # --- Volume features (normalized) ---
    for window in [5, 10, 20]:
        vol_sma = df['Volume'].rolling(window).mean()
        df[f'Volume_Ratio_{window}'] = df['Volume'] / (vol_sma + 1e-10)
    df['Volume_Change'] = df['Volume'].pct_change()

    # OBV normalized (rate of change)
    obv = (np.sign(df['Close'].diff()) * df['Volume']).cumsum()
    for period in [5, 10, 20]:
        df[f'OBV_ROC_{period}'] = obv.pct_change(period)

    # --- Rate of Change ---
    for period in [1, 5, 10, 21, 63]:
        df[f'ROC_{period}'] = df['Close'].pct_change(period)

    # --- Realized Volatility ---
    for window in [5, 10, 21, 63]:
        df[f'Volatility_{window}'] = df['Log_Returns'].rolling(window).std() * np.sqrt(252)

    # --- Volatility ratio (short/long) ---
    df['Vol_Ratio_5_21'] = (df['Log_Returns'].rolling(5).std()) / (df['Log_Returns'].rolling(21).std() + 1e-10)
    df['Vol_Ratio_10_63'] = (df['Log_Returns'].rolling(10).std()) / (df['Log_Returns'].rolling(63).std() + 1e-10)

    # --- Mean reversion signals ---
    for window in [10, 21, 63]:
        rolling_mean = df['Returns'].rolling(window).mean()
        rolling_std = df['Returns'].rolling(window).std()
        df[f'Zscore_{window}'] = (df['Returns'] - rolling_mean) / (rolling_std + 1e-10)

    # --- Momentum (rate of change of rate of change) ---
    df['Momentum_Accel'] = df['ROC_5'] - df['ROC_5'].shift(5)

    # --- Price dispersion ---
    df['High_Low_Ratio'] = df['High'] / (df['Low'] + 1e-10)

    # --- Calendar features (cyclical encoding) ---
    if pd.api.types.is_datetime64_any_dtype(df['Date']):
        df['DayOfWeek_Sin'] = np.sin(2 * np.pi * df['Date'].dt.dayofweek / 5)
        df['DayOfWeek_Cos'] = np.cos(2 * np.pi * df['Date'].dt.dayofweek / 5)
        df['Month_Sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)
        df['DayOfYear_Sin'] = np.sin(2 * np.pi * df['Date'].dt.dayofyear / 252)
        df['DayOfYear_Cos'] = np.cos(2 * np.pi * df['Date'].dt.dayofyear / 252)

    return df


# ─────────────────────────────────────────────
# 2. DATA PREPARATION
# ─────────────────────────────────────────────

def prepare_data(df, target_col='Close', forecast_horizon=63, sequence_length=60,
                 test_ratio=0.2):
    """
    Prepare data with proper train/test split and scaling.
    Uses RobustScaler (better with outliers than MinMaxScaler).
    """
    df = df.copy()

    # Multi-horizon target: average of multiple horizons for stability
    horizons = [forecast_horizon]
    if forecast_horizon >= 42:
        horizons = [
            max(21, forecast_horizon - 21),
            forecast_horizon,
            forecast_horizon + 21,
        ]

    targets = []
    for h in horizons:
        targets.append(df[target_col].shift(-h) / df[target_col] - 1)
    df['Target'] = pd.concat(targets, axis=1).mean(axis=1)

    df = df.dropna()

    # Feature columns (exclude non-features)
    exclude_cols = ['Date', 'Target', 'Open', 'High', 'Low', 'Close', 'Volume']
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    # Train/test split
    split_idx = int(len(df) * (1 - test_ratio))
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    # RobustScaler for features
    feature_scaler = RobustScaler()
    train_features = feature_scaler.fit_transform(train_df[feature_cols])
    test_features = feature_scaler.transform(test_df[feature_cols])

    # Clip extreme values after scaling
    train_features = np.clip(train_features, -5, 5)
    test_features = np.clip(test_features, -5, 5)

    # Scale target for LSTM
    target_scaler = RobustScaler()
    train_target = target_scaler.fit_transform(train_df[['Target']]).ravel()
    test_target_scaled = target_scaler.transform(test_df[['Target']]).ravel()
    test_target_raw = test_df['Target'].values

    test_dates = test_df['Date'].values
    test_close = test_df['Close'].values

    # --- XGBoost data (flat, uses raw target) ---
    xgb_data = {
        'X_train': train_features,
        'X_test': test_features,
        'y_train': train_df['Target'].values,
        'y_test': test_target_raw,
    }

    # --- LSTM data (sequences, uses scaled target) ---
    def create_sequences(features, targets, seq_len):
        X, y = [], []
        for i in range(seq_len, len(features)):
            X.append(features[i - seq_len:i])
            y.append(targets[i])
        return np.array(X), np.array(y)

    X_train_seq, y_train_seq = create_sequences(train_features, train_target, sequence_length)
    X_test_seq, y_test_seq = create_sequences(test_features, test_target_scaled, sequence_length)

    lstm_data = {
        'X_train': X_train_seq,
        'X_test': X_test_seq,
        'y_train': y_train_seq,
        'y_test': y_test_seq,
    }

    return (xgb_data, lstm_data, feature_cols, feature_scaler,
            target_scaler, test_dates, test_close, test_target_raw)


# ─────────────────────────────────────────────
# 3. Bi-LSTM WITH ATTENTION
# ─────────────────────────────────────────────

class Attention(nn.Module):
    """Simple self-attention over LSTM outputs."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1, bias=False),
        )

    def forward(self, lstm_output):
        weights = self.attn(lstm_output)
        weights = torch.softmax(weights, dim=1)
        context = (lstm_output * weights).sum(dim=1)
        return context


class BiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )
        self.attention = Attention(hidden_size * 2)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size * 2),
            nn.Linear(hidden_size * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        out = self.head(context)
        return out.squeeze(-1)


def train_lstm(lstm_data, input_size, epochs=150, batch_size=32, lr=0.0005):
    """Train Bi-LSTM with attention, Huber loss, cosine annealing."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    X_train = torch.FloatTensor(lstm_data['X_train']).to(device)
    y_train = torch.FloatTensor(lstm_data['y_train']).to(device)
    X_test = torch.FloatTensor(lstm_data['X_test']).to(device)

    val_size = int(len(X_train) * 0.15)
    X_val, y_val = X_train[-val_size:], y_train[-val_size:]
    X_train, y_train = X_train[:-val_size], y_train[:-val_size]

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True,
    )

    model = BiLSTMAttention(input_size).to(device)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=30, T_mult=2, eta_min=1e-6
    )

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if (epoch + 1) % 25 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1}/{epochs} | Train: {train_loss/len(train_loader):.6f} | "
                  f"Val: {val_loss:.6f} | LR: {current_lr:.2e}")

        if patience_counter >= 30:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test).cpu().numpy()

    return model, test_pred


# ─────────────────────────────────────────────
# 4. XGBOOST MODEL
# ─────────────────────────────────────────────

def train_xgboost(xgb_data):
    """Train XGBoost with Huber loss (reg:pseudohubererror)."""
    val_size = int(len(xgb_data['X_train']) * 0.15)
    X_train = xgb_data['X_train'][:-val_size]
    y_train = xgb_data['y_train'][:-val_size]
    X_val = xgb_data['X_train'][-val_size:]
    y_val = xgb_data['y_train'][-val_size:]

    model = XGBRegressor(
        n_estimators=2000,
        max_depth=4,
        learning_rate=0.005,
        subsample=0.7,
        colsample_bytree=0.6,
        colsample_bylevel=0.6,
        reg_alpha=1.0,
        reg_lambda=5.0,
        gamma=0.1,
        min_child_weight=10,
        objective='reg:pseudohubererror',
        early_stopping_rounds=100,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    test_pred = model.predict(xgb_data['X_test'])
    print(f"  Best iteration: {model.best_iteration}")
    return model, test_pred


# ─────────────────────────────────────────────
# 5. LIGHTGBM MODEL
# ─────────────────────────────────────────────

def train_lightgbm(xgb_data):
    """Train LightGBM with Huber loss."""
    if not HAS_LGBM:
        return None, np.zeros(len(xgb_data['X_test']))

    val_size = int(len(xgb_data['X_train']) * 0.15)
    X_train = xgb_data['X_train'][:-val_size]
    y_train = xgb_data['y_train'][:-val_size]
    X_val = xgb_data['X_train'][-val_size:]
    y_val = xgb_data['y_train'][-val_size:]

    model = LGBMRegressor(
        n_estimators=2000,
        max_depth=4,
        learning_rate=0.005,
        subsample=0.7,
        colsample_bytree=0.6,
        reg_alpha=1.0,
        reg_lambda=5.0,
        min_child_samples=20,
        objective='huber',
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            __import__('lightgbm').early_stopping(100, verbose=False),
            __import__('lightgbm').log_evaluation(0),
        ],
    )

    test_pred = model.predict(xgb_data['X_test'])
    print(f"  Best iteration: {model.best_iteration_}")
    return model, test_pred


# ─────────────────────────────────────────────
# 6. STACKING ENSEMBLE
# ─────────────────────────────────────────────

def stacking_ensemble(y_true, preds_dict, test_preds_dict):
    """
    Train a Ridge meta-learner on base model predictions.
    Uses time-aware split to avoid leakage.
    No intercept so ensemble is a pure weighted combination of models.
    """
    train_meta = np.column_stack([preds_dict[k] for k in sorted(preds_dict.keys())])
    test_meta = np.column_stack([test_preds_dict[k] for k in sorted(test_preds_dict.keys())])

    split = int(len(train_meta) * 0.7)
    X_meta_train, y_meta_train = train_meta[:split], y_true[:split]

    meta = Ridge(alpha=1.0, fit_intercept=False)
    meta.fit(X_meta_train, y_meta_train)

    weights = meta.coef_ / (meta.coef_.sum() + 1e-10)
    for name, w in zip(sorted(preds_dict.keys()), weights):
        print(f"    {name}: {w:.3f}")

    ensemble_pred = meta.predict(test_meta)
    return meta, ensemble_pred


# ─────────────────────────────────────────────
# 7. EVALUATION
# ─────────────────────────────────────────────

def evaluate(y_true, y_pred, label="Model"):
    """Compute regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    direction_acc = np.mean(np.sign(y_true) == np.sign(y_pred))
    from scipy.stats import spearmanr
    ic, _ = spearmanr(y_true, y_pred)

    print(f"  {label}:")
    print(f"    MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | "
          f"Dir Acc: {direction_acc:.2%} | IC: {ic:.4f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'direction_acc': direction_acc, 'ic': ic}


# ─────────────────────────────────────────────
# 8. VISUALIZATION
# ─────────────────────────────────────────────

def plot_results(test_dates, test_close, y_true, preds, ensemble_pred, forecast_horizon):
    """Enhanced visualization."""
    offset = len(y_true)
    dates = test_dates[-offset:] if len(test_dates) > offset else test_dates
    close = test_close[-offset:] if len(test_close) > offset else test_close

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    # 1. Predicted vs Actual Returns
    ax = axes[0, 0]
    ax.plot(dates, y_true, label='Actual', alpha=0.7, linewidth=1)
    ax.plot(dates, ensemble_pred, label='Ensemble', alpha=0.7, linewidth=1)
    ax.fill_between(dates, y_true, ensemble_pred, alpha=0.1, color='red')
    ax.set_title(f'Predicted vs Actual {forecast_horizon}-Day Returns')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

    # 2. Predicted Price
    ax = axes[0, 1]
    actual_fp = close * (1 + y_true)
    pred_fp = close * (1 + ensemble_pred)
    ax.plot(dates, actual_fp, label='Actual Future Price', alpha=0.7)
    ax.plot(dates, pred_fp, label='Predicted Future Price', alpha=0.7)
    ax.set_title(f'Implied {forecast_horizon}-Day Ahead Price')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

    # 3. Individual model scatter
    ax = axes[0, 2]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    for i, (name, pred) in enumerate(preds.items()):
        ax.scatter(y_true, pred, alpha=0.3, s=10, label=name, color=colors[i % len(colors)])
    lims = [min(y_true.min(), ensemble_pred.min()) * 1.1, max(y_true.max(), ensemble_pred.max()) * 1.1]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect')
    ax.set_xlabel('Actual Returns')
    ax.set_ylabel('Predicted Returns')
    ax.set_title('Scatter: Predicted vs Actual')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # 4. Rolling directional accuracy
    ax = axes[1, 0]
    window = min(50, len(y_true) // 4)
    dir_correct = (np.sign(y_true) == np.sign(ensemble_pred)).astype(float)
    rolling_acc = pd.Series(dir_correct).rolling(window).mean().values
    ax.plot(dates, rolling_acc, label=f'Rolling {window}-day Dir. Accuracy', color='green')
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random (50%)')
    ax.set_title('Rolling Directional Accuracy')
    ax.set_ylim(0.2, 0.9)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

    # 5. Prediction error distribution
    ax = axes[1, 1]
    errors = y_true - ensemble_pred
    ax.hist(errors, bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(x=0, color='red', linestyle='--')
    ax.set_title('Prediction Error Distribution')
    ax.set_xlabel('Error (Actual - Predicted)')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

    # 6. Running MAE
    ax = axes[1, 2]
    abs_errors = np.abs(errors)
    running_mae = np.cumsum(abs_errors) / np.arange(1, len(abs_errors) + 1)
    rolling_mae = pd.Series(abs_errors).rolling(window).mean().values
    ax.plot(dates, running_mae, label='Cumulative MAE', alpha=0.7)
    ax.plot(dates, rolling_mae, label=f'Rolling {window}-day MAE', alpha=0.7)
    ax.set_title('Mean Absolute Error Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n  Results plot saved to: prediction_results.png")


def plot_feature_importance(xgb_model, lgbm_model, feature_cols, top_n=20):
    """Plot combined feature importance from XGBoost + LightGBM."""
    importance = xgb_model.feature_importances_.copy()
    if lgbm_model is not None:
        importance = (importance + lgbm_model.feature_importances_) / 2

    indices = np.argsort(importance)[-top_n:]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(indices)), importance[indices])
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([feature_cols[i] for i in indices])
    ax.set_title(f'Top {top_n} Feature Importances (Avg XGBoost + LightGBM)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("  Feature importance plot saved to: feature_importance.png")


# ─────────────────────────────────────────────
# 9. FUTURE PREDICTION
# ─────────────────────────────────────────────

def predict_future(df, lstm_model, xgb_model, lgbm_model, meta_model,
                   feature_cols, feature_scaler, target_scaler,
                   sequence_length=60):
    """Predict future price using most recent data."""
    device = next(lstm_model.parameters()).device

    df_feat = add_technical_indicators(df)
    df_feat = df_feat.dropna()

    features = feature_scaler.transform(df_feat[feature_cols].iloc[-sequence_length:])
    features = np.clip(features, -5, 5)

    xgb_pred = xgb_model.predict(features[-1:])[0]

    if lgbm_model is not None:
        lgbm_pred = lgbm_model.predict(features[-1:])[0]
    else:
        lgbm_pred = xgb_pred

    lstm_input = torch.FloatTensor(features).unsqueeze(0).to(device)
    lstm_model.eval()
    with torch.no_grad():
        lstm_pred_scaled = lstm_model(lstm_input).cpu().numpy()
    lstm_pred = target_scaler.inverse_transform(lstm_pred_scaled.reshape(-1, 1)).ravel()[0]

    meta_features = np.array([[lstm_pred, lgbm_pred, xgb_pred]])
    ensemble = meta_model.predict(meta_features)[0]

    current_price = df_feat['Close'].iloc[-1]
    predicted_price = current_price * (1 + ensemble)

    return {
        'current_price': current_price,
        'predicted_return': ensemble,
        'predicted_price': predicted_price,
        'lstm_return': lstm_pred,
        'xgb_return': xgb_pred,
        'lgbm_return': lgbm_pred,
        'current_date': df_feat['Date'].iloc[-1],
    }


# ─────────────────────────────────────────────
# 10. MAIN PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(csv_path='stock_data.csv', forecast_horizon=63, sequence_length=60):
    print("=" * 65)
    print(f"Stock Price Prediction V2 ({forecast_horizon} trading days ahead)")
    print("=" * 65)

    print("\n[1/7] Loading data...")
    df = load_csv_with_date(csv_path)
    df = df.sort_values('Date').reset_index(drop=True)
    print(f"  {len(df)} rows: {df['Date'].min().date()} to {df['Date'].max().date()}")
    print(f"  Columns: {list(df.columns)}")

    print("\n[2/7] Engineering stationary features...")
    df_feat = add_technical_indicators(df)
    print(f"  {len(df_feat.columns) - 6} features (all ratios/normalized)")

    print("\n[3/7] Preparing data...")
    (xgb_data, lstm_data, feature_cols, feature_scaler,
     target_scaler, test_dates, test_close, test_target_raw) = \
        prepare_data(df_feat, forecast_horizon=forecast_horizon, sequence_length=sequence_length)
    print(f"  XGBoost  - Train: {xgb_data['X_train'].shape}, Test: {xgb_data['X_test'].shape}")
    print(f"  LSTM     - Train: {lstm_data['X_train'].shape}, Test: {lstm_data['X_test'].shape}")
    print(f"  Features: {len(feature_cols)}")

    print("\n[4/7] Training XGBoost...")
    xgb_model, xgb_pred = train_xgboost(xgb_data)

    print("\n[5/7] Training LightGBM...")
    if HAS_LGBM:
        lgbm_model, lgbm_pred = train_lightgbm(xgb_data)
    else:
        lgbm_model, lgbm_pred = None, xgb_pred.copy()
        print("  Skipped (not installed)")

    print("\n[6/7] Training Bi-LSTM with Attention...")
    lstm_model, lstm_pred_scaled = train_lstm(lstm_data, input_size=len(feature_cols), epochs=150)
    lstm_pred = target_scaler.inverse_transform(lstm_pred_scaled.reshape(-1, 1)).ravel()

    min_len = min(len(lstm_pred), len(xgb_pred), len(lgbm_pred))
    lstm_pred_aligned = lstm_pred[-min_len:]
    xgb_pred_aligned = xgb_pred[-min_len:]
    lgbm_pred_aligned = lgbm_pred[-min_len:]
    y_test = test_target_raw[-min_len:]

    print("\n[7/7] Stacking ensemble...")
    print("  Meta-learner weights:")
    meta_split = int(min_len * 0.7)
    train_preds = {'LSTM': lstm_pred_aligned[:meta_split], 'LightGBM': lgbm_pred_aligned[:meta_split], 'XGBoost': xgb_pred_aligned[:meta_split]}
    test_preds = {'LSTM': lstm_pred_aligned, 'LightGBM': lgbm_pred_aligned, 'XGBoost': xgb_pred_aligned}
    y_train_meta = y_test[:meta_split]
    meta_model, ensemble_pred = stacking_ensemble(y_train_meta, train_preds, test_preds)

    print(f"\n{'='*65}")
    print("EVALUATION RESULTS")
    print(f"{'='*65}")
    evaluate(y_test, lstm_pred_aligned, "LSTM")
    evaluate(y_test, xgb_pred_aligned, "XGBoost")
    if HAS_LGBM:
        evaluate(y_test, lgbm_pred_aligned, "LightGBM")
    metrics = evaluate(y_test, ensemble_pred, "Ensemble (Stacked)")

    print("\n  Generating plots...")
    preds = {'LSTM': lstm_pred_aligned, 'XGBoost': xgb_pred_aligned, 'Ensemble': ensemble_pred}
    if HAS_LGBM:
        preds['LightGBM'] = lgbm_pred_aligned
    plot_results(test_dates, test_close, y_test, preds, ensemble_pred, forecast_horizon)
    plot_feature_importance(xgb_model, lgbm_model, feature_cols)

    print(f"\n{'='*65}")
    print("FUTURE PREDICTION")
    print(f"{'='*65}")
    result = predict_future(df, lstm_model, xgb_model, lgbm_model, meta_model,
                            feature_cols, feature_scaler, target_scaler, sequence_length)
    print(f"  Current Date:     {result['current_date']}")
    print(f"  Current Price:    ${result['current_price']:.2f}")
    print(f"  Horizon:          {forecast_horizon} trading days")
    print(f"  LSTM Return:      {result['lstm_return']:.2%}")
    print(f"  XGBoost Return:   {result['xgb_return']:.2%}")
    if HAS_LGBM:
        print(f"  LightGBM Return:  {result['lgbm_return']:.2%}")
    print(f"  Ensemble Return:  {result['predicted_return']:.2%}")
    print(f"  Predicted Price:  ${result['predicted_price']:.2f}")
    print(f"{'='*65}")

    return {
        'lstm_model': lstm_model,
        'xgb_model': xgb_model,
        'lgbm_model': lgbm_model,
        'meta_model': meta_model,
        'feature_cols': feature_cols,
        'feature_scaler': feature_scaler,
        'target_scaler': target_scaler,
        'result': result,
        'metrics': metrics,
    }


# ─────────────────────────────────────────────
# SAMPLE DATA GENERATOR
# ─────────────────────────────────────────────

def generate_sample_data(filename='stock_data.csv', n_days=1500):
    np.random.seed(42)
    dates = pd.bdate_range(end='2026-03-03', periods=n_days)

    price = 150.0
    prices = []
    for _ in range(n_days):
        ret = np.random.normal(0.0003, 0.015)
        price *= (1 + ret)
        daily_vol = abs(np.random.normal(0.01, 0.005))
        open_p = price * (1 + np.random.normal(0, 0.003))
        high_p = max(open_p, price) * (1 + abs(np.random.normal(0, daily_vol)))
        low_p = min(open_p, price) * (1 - abs(np.random.normal(0, daily_vol)))
        volume = int(np.random.lognormal(17, 0.5))
        prices.append([open_p, high_p, low_p, price, volume])

    df = pd.DataFrame(prices, columns=['Open', 'High', 'Low', 'Close', 'Volume'], index=dates)
    df.index.name = 'Date'
    df = df.round(2)
    df.to_csv(filename)
    print(f"Sample data saved to {filename} ({n_days} days)")
    return df


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == '__main__':
    import os

    csv_file = 'Microsoft.csv'
    if not os.path.exists(csv_file):
        print("No stock_data.csv found. Generating sample data...\n")
        generate_sample_data(csv_file)

    results = run_pipeline(csv_file, forecast_horizon=63)